# Danish Agriculture Agency Playground

This notebook is meant as a playground to get familiar with the [Danish Agriculture Agency](https://landbrugsgeodata.fvm.dk/) different datasets.

In [ ]:
import ee
import geemap
import os
from dotenv import load_dotenv
import numpy as np
load_dotenv()

# Initialize the Earth Engine module.
PROJECT_ID = os.getenv("PROJECT_ID")
ee.Initialize(project = os.getenv("PROJECT_ID"))

# Define map visual parameters
DW_CLASS = {
    'water' : '419bdf', # blue
    'trees' : '397d49', # green
    'grass' : '88b053', # light green
    'flooded_vegetation' : '7a87c6', # greyish blue
    'crops' : 'e49635', # orange
    'shrub_and_scrub' : 'dfc35a', # yellow
    'built' : 'c4281b', # red
    'bare' : 'a59b8f', # grey
    'snow_and_ice' : 'b39fe1', # purple
}

DW_VIZ_PARAMS = {
  'min' : 0,
  'max' : 8,
  'palette' : list(DW_CLASS.values()),
  'bands' : 'label'
}

# B2, B3, B4 are the optical visualization bands, 
# min defines the values that should be mapped to RGB(0) and max the values that should be mapped to RGB(255)
S2_VIZ_PARAMS = {
  'min' : 0,
  'max' : 3000,
  'bands' : ['B4', 'B3', 'B2']
}

# Define Denmark Area
DENMARK = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017').filter(ee.Filter.eq('country_na', 'Denmark')).geometry()

The datasets coming from DAA come in the format of shapefiles. To use the dataset as Earth Engine Object we first need to upload it to [EE Code Editor](https://code.earthengine.google.com/). Geemap function `shp_to_ee`  would have also been an option, but it requires the CRS of our original dataset to be EPSG:4326, which is not the case. Earth Engine only cares about the .shp, .shx, .dbf and .prj fil extensions. When uploading the DAA datasets is important to clarify that our dataset follows `ISO 88591` and not the standard `UTF-8`. 

Once our dataset is uploaded we can now access it as a `FeatureCollection` object.

In [ ]:
YEAR = "25"

dataset = ee.FeatureCollection("projects/" + PROJECT_ID + "/assets/Markblokke" + YEAR)
display(dataset.limit(5))

## Exploring the DAA Dataset

### Markblokke

A Markblok is a continuous geographical area of farmland defined by permanent boundaries in the landscape (like roads, hedges, or streams). Inside a single field block, one or more farmers might have several individual crops (fields). [Reference](https://geodata-info.dk/srv/api/recordsd91b2c99-d9b0-4e6d-b323-20ac80548186#:~:text=Markblokkortet%20er%20et%20digitalt%20markkort,typisk%20permanente%20skel%20i%20landskabet.) The data collected comes from information provided by the farmers which enter the information to apply for EU CAP subsidies, aerial pictures and is reviewed by professional from the agency. 

Our dataset has the following columns:

- **MARKBLOKNR** (String): unique identification number for a specific field block
- **GEOMETRISK** (Float): area without deducting non-eligible area in hectares
- **GB_FRADRAG** (Float): area *not* eligible for standard EU agricultural support (such as large clumps of trees, buildings, or roads) in hectares
- **GB_AREAL** (Float): area eligible for subsidy in hectares
- **MB_TYPE** (String/Enum): type of the field block ([Reference](https://lbst.dk/Media/638467079774681715Brugerguide_marker_og_markblokke.pdf))
    - OMD - Arable land/Rotation land
    - PGR - Permanent grass (grass)
    - PAF - Permanent crops (crops)
    - MIX - Mixed field block
    - VKS - Greenhouses
    - LDP - Sustained/Afforested Woodland
    - ING - No support
    - SOL - Solar Farms

## Total Field Area: How big is the total field area?

Possibilities to calculate this value:
* summing all GEOMETRISK values
* summing all GB_AREAL + GB_FRADRAG values (the sum of this two is not the same as GEOMETRISK why?)
* summing the areas of the features


|  | Feature Collection DAA | Geometrisk Sum | GB Sum | Fradrag Area | GB + Fradrag |
| --- | --- | --- | --- | --- | --- |
| 2023 |  |  |  |  |  |
| 2024 | 26982.90 | 27075.33 | 26171.44 | 54.33 | 26225.77 |
| 2025 | 26925.19 | 27017.62 | 26197.64 | 49.62 | 26247.26 |

In [ ]:
geometrisk_sum = dataset.aggregate_array('GEOMETRISK').reduce(ee.Reducer.sum()).getInfo()
geometrisk_sum_km = ee.Number(geometrisk_sum).divide(pow(10,2)).getInfo()
print(f"Sum of total area of all field blocks:", geometrisk_sum_km )

Total Area: 27017.622499999983


In [ ]:
area_sum = dataset.aggregate_array('GB_AREAL').reduce(ee.Reducer.sum()).getInfo()
area_sum_km = ee.Number(area_sum).divide(pow(10,2)).getInfo()
print(f"Sum of subsidized area of all field blocks:", area_sum_km )

Total Area: 26197.640700000105


In [ ]:
non_subsidized_sum = dataset.aggregate_array('GB_FRADRAG').reduce(ee.Reducer.sum()).getInfo()
non_subsidized_sum_km = ee.Number(non_subsidized_sum).divide(pow(10,2)).getInfo()
print(f"Sum of non-subsidized area of all field blocks:", non_subsidized_sum_km )

Total Area: 49.62230000000107


In [ ]:
def add_area(feature):
    return feature.set('area_m2', feature.geometry().area())

with_area = dataset.map(add_area)
total_area = with_area.aggregate_sum('area_m2').getInfo()

field_area_km = ee.Number(total_area / pow(10,6))
print("Total area som coming from the polygons (km²):" )
display(field_area_km)

In [ ]:
denmark_area_km = ee.Number(DENMARK.area()).divide(1e6)
denmark_field_area = field_area_km.divide(denmark_area_km).multiply(100)

print("Total Area of Denmark (km²):")
display(denmark_area_km)
print("Percentage of Denmark which is field:")
display(denmark_field_area)

Total Field Area in Denmark (km²):


Total Area of Denmark (km²):


Percentage of Denmark which is field:


## Merging with Dynamic World

Loading DW Image with the year mode of Denmark anc removing non field areas from DW image.

In [ ]:
dw_image = ee.Image('projects/' + PROJECT_ID + '/assets/dw_20' + YEAR + '_mode')
display(dw_image)

clipped_img = dw_image.clipToCollection(dataset)

## Out of the entirety of Denmark, how much did DW predict as agriculture?

Is this analysis possible? Does it make sense? Sum of crops+grass+trees does not make sense.
Does it make sense to predict crops outside of DAA defined area? or can that be considered an missclassification?
Currently Denmark includes more than Denmark. It includes all the MMU that are present in Denmark

|  | DW Crop Area | DW Crop Area Inside | DW Crop Area Outside |
| --- | --- | --- | --- |
| 2023 |  |  |  |
| 2024 | 28929.36 | 21001.98 | 7927.37 |
| 2025 | 25529.08 | 18589.29 |  |

Calculating the area for each Dynamic World classification type (Code adpted from [here](https://spatialthoughts.com/2020/06/19/calculating-area-gee/))


In [ ]:
def area_class(dw_class):
    area_image = dw_class.multiply(ee.Image.pixelArea())
    area = area_image.reduceRegion( 
        reducer = ee.Reducer.sum(),
        scale = 10,
        maxPixels = 1e13
    )
    return ee.Number(area.get('label')).divide(pow(10,6))

In [ ]:
dw_crops = dw_image.eq(4)
dw_crops_area = area_class(dw_crops)
print("DW Crops Area:")
display(dw_crops_area)

DW Crops Area:


## How much of the area inside of DAA got classified as each DW class?

|  | Water | Trees | Grass | Flooded Vegetation | Crops | Shrub & Scrub | Built | Bare | Snow & Ice |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2023 |  |  |  |  |  |  |  |  |  |
| 2024 | 59.55 | 2875.20 | 2748.15 | 49.60 | 21001.98 | 59.27 | 240.01 | 12.11 | 1.25 |
| 2025 | 64.06 | 2440.08 | 5470.97 | 27.95 | 18589.29 | 154.01 | 222.01 | 22.65 | 0.89 |

Note: Do we want to take mb_types into consideration?

In [ ]:
# Calculating how much of the DAA got classified as each class
for i in range(9):
    dw_class = clipped_img.eq(i)
    daa_area = area_class(dw_class)
    print(f"DAA Area Classified as", i)
    display(daa_area)

DAA Area Classified as 0


DAA Area Classified as 1


DAA Area Classified as 2


DAA Area Classified as 3


DAA Area Classified as 4


DAA Area Classified as 5


DAA Area Classified as 6


DAA Area Classified as 7


DAA Area Classified as 8


## Visual Representation of DW Intersection with Markblokke

In [ ]:
# img = linked_col.mosaic
overlap_map = geemap.Map(scroll_wheel_zoom = False)
# map.centerObject(s2_img, 12)

# Add DW classification layer
overlap_map.add_layer(
    dw_image,
    DW_VIZ_PARAMS,
    'Dynamic World',
)
# Add DAA polygon area layer
overlap_map.add_layer(
    dataset, 
    {'color': 'white', 'oppacity': 0.2},
    "Polygon Boundaries",
)
overlap_map

In [ ]:
map = geemap.Map(scroll_wheel_zoom = False)

map.add_layer(
    s2_img,
    S2_VIZ_PARAMS, 
    'Sentinel-2 L1C',
    False
)
map.add_layer(
    clipped_img,
    DW_VIZ_PARAMS,
    'Dynamic World',
)
map